In [21]:
# Force CUDA to be 1
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [22]:
import torch
print(torch.cuda.device_count())  

1


In [23]:
!pip install -q --upgrade  bitsandbytes trl

In [24]:
# Check if we are working on T4
# Check if we are working on T4 (Kaggle)
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)

if gpu_info.find('failed') >= 0 or 'NVIDIA-SMI has failed' in gpu_info:
    print('Not connected to a GPU')
    print("On Kaggle: go to Settings (right sidebar) > Accelerator > GPU T4 x2")
else:
    print('GPU info:')
    print(gpu_info)
    if gpu_info.find('Tesla T4') >= 0:
        print("\nSuccess - Connected to a T4")
        # Kaggle T4 sessions give you 2 GPUs - check how many are visible
        t4_count = gpu_info.count('Tesla T4')
        print(f"Number of T4s detected: {t4_count}")
    else:
        print("\nNOT CONNECTED TO a T4")

GPU info:
Fri Sep 11 21:03:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             35W /   70W |    4233MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------------

In [45]:
# Fine tuning evaluation: fine-tuned model ID ---> tokenization ---> quntization ---> load fine-tuned model ---> evaluation
# Imports
import re
import sys
import math
from tqdm.auto import tqdm
import transformers
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from datetime import datetime
import wandb
from trl import SFTTrainer, SFTConfig
import matplotlib.pyplot as plt

In [46]:
# Connect with utils folder
sys.path.append('/kaggle/input/datasets/omarshaarawy/pricer')
# Class Item to validate the returned data and has lots of helper functions
from items import Item

In [27]:
  !git config --global credential.helper store

In [28]:
# Login to Hugging Face
from huggingface_hub import login
# Library to access Hugging Face token from user secrets
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()         
hf_token = user_secrets.get_secret("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

In [29]:
# Download the batches dataset from Hugging Face
username = "omarshaarawy11"
dataset = f"{username}/best_deal_04_fine_tuning"
train, val, test = Item.from_hub(dataset)
items = train + val + test
print(f"Loaded {len(items):,} items")

Loaded 21,091 items


In [30]:
# Configurations
BASE_MODEL = "meta-llama/Llama-3.2-3B"
# The constant project name
PROJECT_NAME = "best-deal"
USERNAME = "omarshaarawy11"
# Name of our fine-model on Hugging Face
HUB_MODEL_NAME = f"{USERNAME}/best-deal-2026-09-10_18.41.06"
SNAPSHOT = "776421863beaeccbfd320c41b69c7fab10095ca6"
# Hyper parameters
# QLoRA paramters
# Quantization to 4 bits
QUANT_4_BIT = True

In [31]:
# Quantization configurations
# For 4 bits quantization
if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    # Make it float16 when working in Kaggle  
    bnb_4bit_compute_dtype=torch.float16,
    # nf4 stands for normal float 4
    bnb_4bit_quant_type="nf4")
# For 8 bits quantization
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )

In [32]:
# Tokenization
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Quantization
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16
)

# Prepare the 4-bit model for QLoRA training
base_model = prepare_model_for_kbit_training(base_model)

base_model.generation_config.pad_token_id = tokenizer.pad_token_id




Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [33]:
# Invistigate base model
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

In [34]:
# Check the amount of memory that model occupies in RAM or GPU memory
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.2f} GB")

Memory footprint: 2.99 GB


In [35]:
# Load fine-tuned model
fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=SNAPSHOT)

In [36]:
# Invistigate fine-tuned model
fine_tuned_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.L

In [37]:
# Check the amount of memory that model occupies in RAM or GPU memory
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e9:,.2f} GB")

Memory footprint: 3.00 GB


In [38]:
# Test dataset
def fine_tuned_llama(item):
    inputs = tokenizer(
        item.prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output_ids = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=3
        )
    prompt_len = inputs["input_ids"].shape[1]
    # Trim output
    generated_ids = output_ids[0, prompt_len:]
    # Decode output
    output = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )
    # Extract output's number only
    match = re.search(r"\d+(?:\.\d+)?", output)
    return match.group() if match else "0"

In [39]:
# Test single datapointa
fine_tuned_llama(items[0])

'10'

In [47]:
# Evaluation
errors = []
for item in tqdm(test[:2000], total=2000, desc="Evaluating model"):
    prediction = fine_tuned_llama(item)
    match = re.search(r"\d+(?:\.\d+)?", prediction)
    prediction = float(match.group()) if match else 0.0
    error = abs(prediction - item.price)
    errors.append(error)
average_error = sum(errors) / len(errors)
print(f"\nAverage Error: ${average_error:.2f}")

Evaluating model:   0%|          | 0/2000 [00:00<?, ?it/s]


Average Error: $24.94
